# Furniture Sales – Restocking Recommendation Model
**Algorithm:** Random Forest Classifier  
**Labels:** Derived from inventory/sales threshold rules — `Stockout Risk = 1`, others = `0`  
> **Note:** Because labels are built from the same features used for training, scores will be very high by design. For production use, replace `restock_needed` with real historical restock decisions.

In [ ]:
# ============================================================
# Step 0: Imports & global settings
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay,
    precision_recall_curve,
    PrecisionRecallDisplay,
    average_precision_score,
    log_loss,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE   = 42
RESTOCK_THRESH = 0.70    # flag products where restock probability >= 70%
DATA_PATH      = 'Furniture2.csv'

## 1. Load & Inspect Data

In [ ]:
# ============================================================
# Step 1: Load dataset
# ============================================================
df = pd.read_csv(DATA_PATH)

print(f'Shape       : {df.shape[0]} rows x {df.shape[1]} cols')
print(f'Columns     : {df.columns.tolist()}')
print(f'Missing     : {df.isnull().sum().sum()}')
print(f'Duplicates  : {df.duplicated().sum()}')
display(df.head(5))

## 2. Feature Engineering & Label Construction

In [ ]:
# ============================================================
# Step 2: Build derived features
# ============================================================

# Numerical derived features
df['profit_margin']         = df['profit'] / df['price']
df['cost_price_ratio']      = df['cost']   / df['price']
df['inventory_sales_ratio'] = df['inventory'] / (df['sales'] + 1)   # +1 avoids division by zero
df['sales_inventory_ratio'] = df['sales']     / (df['inventory'] + 1)

print('Derived features created:', ['profit_margin', 'cost_price_ratio',
                                     'inventory_sales_ratio', 'sales_inventory_ratio'])

In [ ]:
# ============================================================
# Step 3: Inventory risk classification (rule-based)
# ============================================================
sales_high = df['sales'].quantile(0.75)
sales_low  = df['sales'].quantile(0.25)
inv_high   = df['inventory'].quantile(0.75)
inv_low    = df['inventory'].quantile(0.25)

print(f'Sales  quantiles  →  Q25: {sales_low}  |  Q75: {sales_high}')
print(f'Inventory quantiles →  Q25: {inv_low}  |  Q75: {inv_high}')

def classify_inventory(row):
    if row['inventory'] >= inv_high and row['sales'] <= sales_low:
        return 'Overstock Risk'   # high stock, low demand
    elif row['inventory'] <= inv_low and row['sales'] >= sales_high:
        return 'Stockout Risk'    # low stock, high demand
    return 'Normal'

df['inventory_risk'] = df.apply(classify_inventory, axis=1)
print('\nInventory risk distribution:')
display(df['inventory_risk'].value_counts())

In [ ]:
# ============================================================
# Step 4: Build binary label
# Stockout Risk = 1 (restock needed), all others = 0
# ============================================================
df['restock_needed'] = (df['inventory_risk'] == 'Stockout Risk').astype(int)

print('Label distribution:')
print(df['restock_needed'].value_counts().to_string())
print(f'\nPositive (restock) rate: {df["restock_needed"].mean():.2%}')

# Visualise label imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['restock_needed'].value_counts().plot(
    kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'], edgecolor='white'
)
axes[0].set_title('Label Distribution (count)')
axes[0].set_xticklabels(['No Restock (0)', 'Restock (1)'], rotation=0)
axes[0].set_ylabel('Count')

axes[1].pie(
    df['restock_needed'].value_counts(),
    labels=['No Restock', 'Restock'],
    autopct='%1.1f%%',
    colors=['#4C72B0', '#DD8452'],
    startangle=90
)
axes[1].set_title('Label Distribution (%)')

plt.suptitle('Class Imbalance Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Prepare Features & Encode Categoricals

In [ ]:
# ============================================================
# Step 5: Feature selection & encoding
# ============================================================
FEATURE_COLS = [
    # Numerical features
    'inventory',
    'sales',
    'inventory_sales_ratio',
    'sales_inventory_ratio',
    'delivery_days',
    'profit_margin',
    'price',
    'cost_price_ratio',
    # Categorical features (will be label-encoded)
    'category',
    'season',
    'location',
    'store_type',
    'material',
]

CAT_COLS = ['category', 'season', 'location', 'store_type', 'material']

X = df[FEATURE_COLS].copy()
y = df['restock_needed']

# Label encode categorical columns
encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le    # save encoder for future inference

print('Feature matrix shape:', X.shape)
display(X.head(3))

## 4. Train / Test Split

In [ ]:
# ============================================================
# Step 6: Stratified 80/20 split (preserves class ratio)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,    # ensures both splits have same positive rate
)

print(f'Training set  : {X_train.shape[0]} samples  |  positive rate: {y_train.mean():.2%}')
print(f'Test set      : {X_test.shape[0]} samples  |  positive rate: {y_test.mean():.2%}')

## 5. Hyperparameter Tuning (GridSearchCV)

In [ ]:
# ============================================================
# Step 7: GridSearchCV with 5-fold stratified cross-validation
# Scoring: F1  (better than accuracy for imbalanced labels)
# ============================================================
param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 10, 20],
    'min_samples_split': [2, 5],
    'class_weight':      ['balanced', None],  # 'balanced' compensates for class imbalance
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print('\nBest parameters :', grid_search.best_params_)
print(f'Best CV F1 score : {grid_search.best_score_:.4f}')

best_model = grid_search.best_estimator_

## 6. Model Evaluation — Comprehensive Metrics

| Metric | What it measures | Best for |
|---|---|---|
| **Accuracy** | Overall correct predictions | Balanced classes only |
| **Precision** | Of flagged items, how many truly need restock | Minimising wasted restock cost |
| **Recall** | Of actual stockouts, how many were caught | Minimising missed stockouts ← priority |
| **F1 Score** | Harmonic mean of Precision & Recall | Imbalanced classes |
| **ROC-AUC** | Overall ranking / discrimination quality | General model quality |
| **PR-AUC** | Precision-Recall tradeoff | Imbalanced classes ← more informative than ROC |
| **Log Loss** | How well-calibrated the probabilities are | Probability output quality |
| **MCC** | Balanced single score across all 4 confusion cells | Imbalanced classes |

In [ ]:
# ============================================================
# Step 8: Compute all evaluation metrics
# ============================================================
y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

metrics = {
    'Accuracy':          accuracy_score(y_test, y_pred),
    'Precision':         precision_score(y_test, y_pred, zero_division=0),
    'Recall':            recall_score(y_test, y_pred, zero_division=0),
    'F1 Score':          f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC':           roc_auc_score(y_test, y_proba),
    'PR-AUC (Avg Prec)': average_precision_score(y_test, y_proba),
    'Log Loss':          log_loss(y_test, y_proba),
    'MCC':               matthews_corrcoef(y_test, y_pred),
}

notes = {
    'Accuracy':          'misleading on imbalanced data',
    'Precision':         'of predicted Restock, how many are correct',
    'Recall':            'of actual Restock, how many are caught  <- priority',
    'F1 Score':          'harmonic mean of Precision & Recall',
    'ROC-AUC':           'overall ranking quality  (1.0 = perfect)',
    'PR-AUC (Avg Prec)': 'better than ROC for imbalanced labels',
    'Log Loss':          'probability calibration  (lower = better)',
    'MCC':               'balanced score  (-1 worst | 0 random | 1 best)',
}

print('=' * 68)
print('  Comprehensive Evaluation Metrics')
print('=' * 68)
print(f"  {'Metric':<22} {'Value':>8}   Note")
print('-' * 68)
for name, val in metrics.items():
    print(f"  {name:<22} {val:>8.4f}   {notes[name]}")
print('=' * 68)

print('\nFull Classification Report:')
print(classification_report(y_test, y_pred, target_names=['No Restock', 'Restock']))

In [ ]:
# ============================================================
# Step 9: Plot — Confusion Matrix | ROC Curve | PR Curve
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# Confusion Matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['No Restock', 'Restock'],
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

# ROC Curve
RocCurveDisplay.from_predictions(
    y_test, y_proba, ax=axes[1], color='#4C72B0',
    name=f"Random Forest (AUC={metrics['ROC-AUC']:.3f})"
)
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random baseline')
axes[1].set_title('ROC Curve')
axes[1].legend(fontsize=8)

# PR Curve
PrecisionRecallDisplay.from_predictions(
    y_test, y_proba, ax=axes[2], color='#DD8452',
    name=f"Random Forest (AP={metrics['PR-AUC (Avg Prec)']:.3f})"
)
baseline = y_test.mean()
axes[2].axhline(baseline, color='k', linestyle='--', linewidth=0.8,
                label=f'Random baseline ({baseline:.2f})')
axes[2].set_title('Precision-Recall Curve')
axes[2].legend(fontsize=8)

plt.suptitle('Model Evaluation – Confusion Matrix | ROC | PR Curve',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Step 10: Plot — All metrics bar chart
# Red  = priority metrics for imbalanced data
# Blue = general reference metrics
# ============================================================
priority = {'Recall', 'F1 Score', 'PR-AUC (Avg Prec)', 'MCC'}
bar_metrics = {k: v for k, v in metrics.items() if k != 'Log Loss'}
colors = ['#C44E52' if k in priority else '#4C72B0' for k in bar_metrics]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(list(bar_metrics.keys()), list(bar_metrics.values()),
               color=colors, edgecolor='white')
ax.set_xlim(0, 1.18)
ax.set_xlabel('Score')
ax.set_title(
    'Evaluation Metrics Overview\n(red = priority metrics for imbalanced data)',
    fontweight='bold'
)
for bar, val in zip(bars, bar_metrics.values()):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)
ax.axvline(1.0, color='gray', linestyle='--', linewidth=0.7)

# Log Loss annotation (different scale — lower is better)
ax.text(1.10, -0.8,
        f"Log Loss\n{metrics['Log Loss']:.4f}\n(lower = better)",
        ha='center', fontsize=8, color='#555555',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#f0f0f0', edgecolor='gray'))

plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
# ============================================================
# Step 11: Feature importance analysis
# ============================================================
importance_df = (
    pd.DataFrame({
        'feature':    X.columns,
        'importance': best_model.feature_importances_,
    })
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

display(importance_df)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=importance_df, x='importance', y='feature',
    hue='feature', palette='Blues_r', legend=False
)
plt.title('Feature Importance – Restocking Model', fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 8. Restocking Recommendation List

In [ ]:
# ============================================================
# Step 12: Score all records & generate restock list
# Threshold: restock_proba >= RESTOCK_THRESH (default 70%)
# ============================================================
df_out = df.copy()
df_out['restock_proba'] = best_model.predict_proba(X)[:, 1]
df_out['restock_flag']  = (df_out['restock_proba'] >= RESTOCK_THRESH).astype(int)

restock_list = (
    df_out[df_out['restock_flag'] == 1]
    .sort_values('restock_proba', ascending=False)
    [[
        'category', 'material', 'location', 'store_type', 'season',
        'inventory', 'sales', 'inventory_sales_ratio',
        'profit_margin', 'delivery_days', 'restock_proba',
    ]]
    .reset_index(drop=True)
)

print(f'Products flagged for restocking (probability >= {RESTOCK_THRESH:.0%}): {len(restock_list)}')
display(restock_list.head(20))

restock_list.to_csv('restock_list.csv', index=False)
print('Saved → restock_list.csv')

## 9. Category-Level Urgency Breakdown

In [ ]:
# ============================================================
# Step 13: Restock urgency summary by category
# ============================================================
restock_by_category = (
    df_out.groupby('category')
    .agg(
        avg_restock_proba=('restock_proba', 'mean'),
        flagged_count=('restock_flag',  'sum'),
        total_count=('restock_flag',  'count'),
    )
)
restock_by_category['flag_rate_%'] = (
    restock_by_category['flagged_count'] / restock_by_category['total_count'] * 100
).round(1)
restock_by_category = restock_by_category.sort_values('avg_restock_proba', ascending=False)

display(restock_by_category)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

restock_by_category['avg_restock_proba'].plot(
    kind='bar', ax=axes[0],
    color=sns.color_palette('Reds_r', len(restock_by_category))
)
axes[0].set_title('Avg Restock Probability by Category')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Avg Probability')
axes[0].tick_params(axis='x', rotation=30)

restock_by_category['flag_rate_%'].plot(
    kind='bar', ax=axes[1],
    color=sns.color_palette('Oranges_r', len(restock_by_category))
)
axes[1].set_title('Restock Flag Rate by Category (%)')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Flag Rate (%)')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Category-Level Inventory Urgency', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Summary

### Key Results
- **Positive rate:** 7.76% of products flagged as Stockout Risk
- **Priority metric — Recall:** catching missed stockouts is more critical than false alarms
- **Top features:** `sales_inventory_ratio`, `inventory_sales_ratio`, `sales`, `inventory` drive almost all decisions; categorical features (category, season, location) have minimal weight
- **Category urgency:** Chair > Sofa > Bed > Desk > Dining Table

### Metric Interpretation Guide
| Metric | Your result | Interpretation |
|---|---|---|
| Recall | 1.00 | All actual stockouts caught |
| Precision | 1.00 | No false alarms |
| PR-AUC | 1.00 | Perfect precision-recall tradeoff |
| Log Loss | ~0.009 | Probabilities are well-calibrated |
| MCC | 1.00 | Perfect balanced score |

> **Reminder:** Scores are high because labels were derived from the same inventory/sales rules used as features. Replace `restock_needed` with real historical decisions to get more generalisable results.